### Install Dependencies

In [0]:
# Install Snowflake Connector and Snowpark packages
%pip install snowflake-snowpark-python
%pip install snowflake-connector-python
%pip install snowflake-sqlalchemy


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/170.6 kB ? eta -:--:--
     ━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.7/170.6 kB 1.2 MB/s eta 0:00:01
     ━━━━━━━━━━━━━╸━━━━━━━━━━━━━━━━━━━━━━━━ 61.4/170.6 kB 856.7 kB/s eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━ 102.4/170.6 kB 923.3 kB/s eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━ 163.8/170.6 kB 1.3 MB/s eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.6/170.6 kB 1.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/74.8 kB ? eta -:--:--
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━ 71.7/74.8 kB 11.5 MB/s eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.8/74.8 kB 1.9 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of cryptography to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/1.8 MB ? eta -:--:--
   ━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.2/1.8 MB 8.1 MB/s eta 0:0

In [0]:
%restart_python

## 
### **Import and Create Session**

In [0]:
	from snowflake.snowpark import Session
	from snowflake.snowpark.functions import col
#  Connection Parameters
connection_parameters = {
  "account": "sqishot-fa68768", 
    "user": "poojashree",
    "password": "Poojashree@307",
    "warehouse": "COMPUTE_WH",
    "database": "LOG_ANALYTICS",
    "schema": "PUBLIC",
}
# Create session
session = Session.builder.configs(connection_parameters).create()
print(" Snowpark session created successfully!")


 Snowpark session created successfully!


### Read the File from Azure Stage

In [0]:
df_stage = session.read.options({
    "field_delimiter": ",",
    "skip_header": 1,
    "FIELD_OPTIONALLY_ENCLOSED_BY": '"'
}).csv("@RETAIL_DB.SALES_STAGE.azure_stage/Retail_Sales__500_rows__Preview.csv")
# Show sample data
df_stage.show(5)


Option 'inferSchema' is aliased to 'INFER_SCHEMA'. You may see unexpected behavior. Please refer to format specific options for more information


------------------------------------------------------------------------------------------------------------------------------------------------------------------
|"c1"          |"c2"        |"c3"     |"c4"      |"c5"           |"c6"   |"c7"     |"c8"     |"c9"             |"c10"       |"c11"  |"c12"  |"c13"    |"c14"     |
------------------------------------------------------------------------------------------------------------------------------------------------------------------
|ORD-5F8D6F0C  |2024-10-08  |2024-10  |CUST1000  |Ananya Sharma  |India  |South    |Mumbai   |Office Supplies  |Paper       |9      |0.00   |2700.0   |780.43    |
|ORD-BF0078E4  |2024-08-11  |2024-08  |CUST1001  |Aarav Iyer     |India  |Central  |Lucknow  |Technology       |Networking  |4      |0.15   |27200.0  |4135.60   |
|ORD-86CD58A3  |2024-06-12  |2024-06  |CUST1002  |Arjun Sharma   |USA    |East     |Kolkata  |Furniture        |Tables      |4      |0.10   |31500.0  |5676.96   |
|ORD-FB0CD2D9  |2024-1

### **Write Data into a Table**

In [0]:
# Define your target table name
table_name = "RAW_SALES"

# Write to table (overwrite or append)
df_stage.write.mode("overwrite").save_as_table(table_name)

print(f" Data successfully written into table: {table_name}")


 Data successfully written into table: RAW_SALES


### **Validate Ingestion**

In [0]:
df = session.table("RAW_SALES")
print(f"Total rows loaded: {df.count()}")

df.limit(5).show()


Total rows loaded: 25
------------------------------------------------------------------------------------------------------------------------------------------------------------------
|"c1"          |"c2"        |"c3"     |"c4"      |"c5"           |"c6"   |"c7"     |"c8"     |"c9"             |"c10"       |"c11"  |"c12"  |"c13"    |"c14"     |
------------------------------------------------------------------------------------------------------------------------------------------------------------------
|ORD-5F8D6F0C  |2024-10-08  |2024-10  |CUST1000  |Ananya Sharma  |India  |South    |Mumbai   |Office Supplies  |Paper       |9      |0.00   |2700.0   |780.43    |
|ORD-BF0078E4  |2024-08-11  |2024-08  |CUST1001  |Aarav Iyer     |India  |Central  |Lucknow  |Technology       |Networking  |4      |0.15   |27200.0  |4135.60   |
|ORD-86CD58A3  |2024-06-12  |2024-06  |CUST1002  |Arjun Sharma   |USA    |East     |Kolkata  |Furniture        |Tables      |4      |0.10   |31500.0  |5676.96   |


In [0]:
columns = [
    "OrderID","OrderDate","MonthOfSale","CustomerID","CustomerName",
    "Country","Region","City","Category","Subcategory",
    "Quantity","Discount","Sales","Profit"
]

df_stage = session.read.options({"field_delimiter": ",", "skip_header": 1}).csv(
    "@RETAIL_DB.SALES_STAGE.azure_stage/Retail_Sales__500_rows__Preview.csv"
)

df_stage = df_stage.to_df(*columns)
df_stage.show()


------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|"ORDERID"     |"ORDERDATE"  |"MONTHOFSALE"  |"CUSTOMERID"  |"CUSTOMERNAME"  |"COUNTRY"  |"REGION"  |"CITY"   |"CATEGORY"       |"SUBCATEGORY"  |"QUANTITY"  |"DISCOUNT"  |"SALES"  |"PROFIT"  |
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|ORD-5F8D6F0C  |2024-10-08   |2024-10        |CUST1000      |Ananya Sharma   |India      |South     |Mumbai   |Office Supplies  |Paper          |9           |0.00        |2700.0   |780.43    |
|ORD-BF0078E4  |2024-08-11   |2024-08        |CUST1001      |Aarav Iyer      |India      |Central   |Lucknow  |Technology       |Networking     |4           |0.15        |27200.0  |4135.60   |
|ORD-86CD58A3  |2024-06-12   |2024-

### **Transform & Model via Snowpark**